## Extraccciones API web Gestion de Fugas HWM Global & Hidromejoras

Interaccion con la API mediante request. Esta API no responde en JSON sino en XML
- Cabecera direciones API: https://api.permanetweb-grupomejoras.com/


In [ ]:
# Cargo librerias

import requests
import json
import xml.etree.ElementTree as ET

#Acceso al SO
import sys
import os
from datetime import datetime, timedelta
import time

# Pandas para tablas y csv
import pandas as pd
import io

# Analisis matematico
import numpy as np
import scipy.io.wavfile as wav

# Graficos
import matplotlib.pyplot as plt

# Cargar las variables del archivo .env
from dotenv import load_dotenv
load_dotenv()

## Consulta de estado actual loggers asociados a una cuenta

La respuesta tiene dos partes diferenciadas, que se deben separar en dos tablas, la primera es un mero resumen de un solo registro.

La segunda tiene tanto los detalles de los loggers, como el registro de valores en del último mensaje de cada uno.


In [ ]:
## Consulta de ultimo estado actual de los loggers en la cuenta

# Nueva URL del endpoint
url = "https://api.permanetweb-grupomejoras.com/datagate/api/loggerapi.ashx"

payload = {
    'action': 'getloggers',
    'username': os.getenv("API_USER"),
    'password': os.getenv("API_PASSWORD"),
    'software': 'APIDocumentation'
}

# Realizo la petición GET enviando los parámetros
response = requests.get(url, params=payload)

if response.status_code == 200:
    try:
        root = ET.fromstring(response.text)
        
        # --- DETALLE DE LOS LOGGERS (<logger>) ---
        lista_loggers = []
        
        for logger in root.findall('logger'):
            fila_logger = {}
            
            # Extraemos las propiedades directas de este dispositivo
            for child in logger:
                if child.tag != 'channels':
                    fila_logger[child.tag] = child.text if child.text is not None else ""
            
            # Procesamos subcanales horizontalmente
            channels_node = logger.find('channels')
            if channels_node is not None:
                for channel in channels_node.findall('channel'):
                    ch_num = channel.attrib.get('number', '')
                    last_val = channel.find('lastValue')
                    
                    if last_val is not None and last_val.text:
                        fila_logger[f'channel_{ch_num}_lastValue'] = last_val.text
                    else:
                        fila_logger[f'channel_{ch_num}_lastValue'] = ""
            
            lista_loggers.append(fila_logger)
        
        # Convertimos la lista de diccionarios a DataFrame
        df_detalle = pd.DataFrame(lista_loggers)
        
        # Guardar tabla de inventario
        if not df_detalle.empty:
            df_detalle.to_csv("estado_loggers.csv", index=False, encoding="utf-8-sig")
            print(f"Se han procesado {len(df_detalle)} aparatos y guardado en 'estado_loggers.csv'.")
        else:
            print("No se encontraron registros individuales de loggers.")
            
    except Exception as e:
        print(f"Error al procesar el XML: {e}")
else:
    print(f"Error en la petición. Código de estado: {response.status_code}")

### Creo registro incremental con valores de ultima llamada

Tomo el resultado de la consulta anterior para crear el nuevo registro incremental, se podría fundir en una.

Aprovecho las coordenadas para simplificar usos posteriores.

In [ ]:
# --- CONFIGURACIÓN DE COLUMNAS DINDÁMICAS ---
# Obtenemos la lista total de columnas actuales
columnas_totales = list(df_detalle.columns)

# Calculamos las posiciones solicitadas (restando 1 para usar el índice 0 de Python)
# Campos: 1, 2, 3, 5, 12, 13, 21, 23, 26, 27
indices_fijos = [0, 1, 2, 4, 11, 12, 20, 22, 25, 26]

# Extraemos los nombres de esas columnas fijas
columnas_seleccionadas = [columnas_totales[i] for i in indices_fijos if i < len(columnas_totales)]

# Añadimos los nombres de los 3 últimos campos de la tabla
columnas_seleccionadas.extend(columnas_totales[-3:])

# --- FILTRADO DEL DATAFRAME ---
# Creamos el nuevo DataFrame solo con los campos elegidos
df_incremental = df_detalle[columnas_seleccionadas]

# --- GUARDADO INCREMENTAL ---
nombre_archivo_incremental = "registro_incremental.csv"

# Verificamos si el archivo ya existe para saber si ponemos cabecera (header) o no
archivo_existe = os.path.exists(nombre_archivo_incremental)

# Guardamos usando modo append ('a')
df_incremental.to_csv(
    nombre_archivo_incremental, 
    mode='a', 
    index=False, 
    header=not archivo_existe,  # True si es nuevo (crea cabecera), False si ya existe
    encoding="utf-8-sig"
)

print(f"¡Datos añadidos con éxito de forma incremental en '{nombre_archivo_incremental}'!")
print(f"Columnas guardadas ({len(columnas_seleccionadas)} en total):", columnas_seleccionadas)

# Recuperacion de grabaciones de sonido

La API utiliza dos interfaces diferentes:
- para recuperar la tabla indice de grabaciones: recordingsapi.ashx
- para descargar los archivos de audio: getrecordingsapi.ashx


### Obtengo lista de grabaciones del dia anterior
Comenzamos con la primera para obtener la lista de grabaciones. Si no indicamos un valor ID de mensaje a partir del cual obtener la tabla, o un rango de fechas inicial y/o final tomará las PRIMERAS desde el inicio de registros. En cualquier caso, la tabla estará limitada a 1000 registros.

La respuesta la da en dos tablas, la primera simplemente es la versión de la API, no merece la pena conservarla.

In [ ]:
## Extraigo la lista de grabaciones registradas

# Nueva URL del endpoint
url = "https://api.permanetweb-grupomejoras.com/datagate/api/recordingsapi.ashx"

# Calculo la fecha del dia anterior a la consulta
hoy = datetime.now()
ayer = hoy - timedelta(days=1)
fecha_inicio = ayer.strftime("%Y-%m-%d")
fecha_fin = hoy.strftime("%Y-%m-%d")

payload = {
    'action': 'getregrecord',
    'username': os.getenv("API_USER"),
    'password': os.getenv("API_PASSWORD"),
    'StartDate': fecha_inicio,
    'EndDate': fecha_fin,
    'software': 'APIDocumentation'
}

# Realizo la petición GET enviando los parámetros
response = requests.get(url, params=payload)

if response.status_code == 200:
    try:
        root = ET.fromstring(response.text)
        
        # --- TABLA ÚNICA: Histórico de Grabaciones ---
        lista_grabaciones = []
        
        for rec in root.findall('recording'):
            fila_grabacion = {}
            for child in rec:
                fila_grabacion[child.tag] = child.text if child.text is not None else ""
            
            if fila_grabacion:
                lista_grabaciones.append(fila_grabacion)
        
        # --- CREAR Y EXPORTAR EL CSV ---
        df_detalle = pd.DataFrame(lista_grabaciones)
        
        if not df_detalle.empty:
            nombre_archivo = "lista_grabaciones.csv"
            
            # 1. Comprobar si el archivo ya existe en la carpeta
            archivo_existe = os.path.exists(nombre_archivo)
            
            # 2. Guardar en modo incremental ('a')
            df_detalle.to_csv(
                nombre_archivo, 
                mode='a', 
                index=False, 
                header=not archivo_existe,  # Solo escribe columnas si el archivo es nuevo
                encoding="utf-8-sig"
            )
            
            if archivo_existe:
                print(f"¡Éxito! Se añadieron {len(df_detalle)} nuevos registros a '{nombre_archivo}'.")
            else:
                print(f"¡Éxito! Archivo '{nombre_archivo}' creado por primera vez con {len(df_detalle)} registros.")
                
            print("\nÚltimos registros añadidos:")
            print(df_detalle.tail(3))
        else:
            print("\nNo se encontraron datos nuevos de grabaciones dentro del XML.")
            
    except Exception as e:
        print(f"Error al procesar el XML de grabaciones: {e}")
else:
    print(f"Error en la petición. Código de estado: {response.status_code}")

Los detectores devuelven registros de tipo 2 (histograma) o de tipo 3 (audio o transitorio).

Conviene hacer una tabla acumulativa y consultar para la última fecha (si se tienen menos de 1000 aparatos que puedan fugar y enviar audio el mismo día).

### Uso la lista para descargar de las grabaciones de audio del dia anterior

Debe indicarse el ID de cada grabacion en cada consulta. Devuelve formato wav para el tipo 3 y XML para el tipo 2.


In [ ]:
# Filtro la respuesta anterior para extraer solo audios (.wav) usando el DataFrame de lista grabaciones

# Nueva URL del endpoint
url = "https://api.permanetweb-grupomejoras.com/datagate/api/getrecordingsapi.ashx"

if not df_detalle.empty and 'recordingType' in df_detalle.columns:
    # Filtramos el DataFrame para quedarnos solo con las filas de tipo 3
    df_filtrado = df_detalle[df_detalle['recordingType'] == '3']
    
    if not df_filtrado.empty:
        print(f"\nIniciando descarga de {len(df_filtrado)} grabaciones tipo 3...", flush=True)
        
        # Iteramos sobre las filas filtradas
        for _, fila in df_filtrado.iterrows():
            id_original = fila['id']
            
            # Validamos que el ID exista y no esté vacío
            if pd.notna(id_original) and id_original != "":
                
                # Convertimos de forma segura a un número entero y luego a string para limpiar decimales como '.0'
                try:
                    id_grabacion = str(int(float(id_original)))
                except ValueError:
                    id_grabacion = str(id_original).strip()
                
                # Definimos el nombre dinámico del archivo usando su id_grabacion
                nombre_archivo = f"{id_grabacion}_audiofuga.wav"
                
                payload_audio = {
                    'action': 'getwavrecord',
                    'username': os.getenv("API_USER"),
                    'password': os.getenv("API_PASSWORD"),
                    'software': 'APIDocumentation',
                    'id': id_grabacion
                }
                
                print(f"\nSolicitando audio para el ID: {id_grabacion}...", flush=True)
                
                response_audio = None
                try:
                    response_audio = requests.get(url, params=payload_audio, timeout=15)
                    
                    if response_audio.status_code == 200:
                        # Detectamos si la respuesta es un XML o el audio directo
                        content_type = response_audio.headers.get('Content-Type', '').lower()
                        
                        if 'xml' in content_type:
                            # CASO A: El audio viene dentro de un XML (ej: codificado en Base64)
                            root_audio = ET.fromstring(response_audio.text)
                            audio_node = root_audio.find('.//audio') or root_audio.find('.//data')
                            
                            if audio_node is not None and audio_node.text:
                                datos_binarios = base64.b64decode(audio_node.text)
                                with open(nombre_archivo, "wb") as archivo_wav:
                                    archivo_wav.write(datos_binarios)
                                print(f"-> ¡Éxito! Archivo extraído del XML y guardado como '{nombre_archivo}'", flush=True)
                            else:
                                print(f"-> No se encontró la etiqueta de audio en el XML para el ID {id_grabacion}.", flush=True)
                                
                        else:
                            # CASO B: La API devuelve el archivo binario directamente (.wav crudo)
                            with open(nombre_archivo, "wb") as archivo_wav:
                                archivo_wav.write(response_audio.content)
                            print(f"-> ¡Éxito! Archivo binario directo guardado como '{nombre_archivo}'", flush=True)
                            
                    else:
                        print(f"-> Error en la petición para ID {id_grabacion}. Estado: {response_audio.status_code}", flush=True)
                        
                except Exception as e:
                    print(f"-> Error al procesar la conexión del ID {id_grabacion}: {e}", flush=True)
                
                finally:
                    # Liberamos explícitamente la conexión del objeto response si se creó
                    if response_audio is not None:
                        response_audio.close()
                
                # --- CONTROL DE TRÁFICO (RATE LIMIT) ---
                # Espera de 1 segundo entre peticiones para no saturar el servidor de la API
                print("Esperando 1 segundo antes de la siguiente consulta...", flush=True)
                time.sleep(1)
                
        print("\n¡Bucle finalizado por completo!", flush=True)
    else:
        print("\nNo se encontraron grabaciones con recordingType = 3 en este lote.", flush=True)


### Uso la lista también para descargar histogramas (grabaciones tipo 2)

Hago algo similar para obtener los histogramas completos del dia anterior (grabaciones de tipo 2) en la lista diaria.

In [ ]:
# (HISTOGRAMAS): Filtrar y procesar datos tipo aqualog (tipo 2) usando el DataFrame
if not df_detalle.empty and 'recordingType' in df_detalle.columns:
    # Filtramos el DataFrame para quedarnos solo con las filas de tipo 2
    df_filtro = df_detalle[df_detalle['recordingType'] == '2']
    
    if not df_filtro.empty:
        print(f"\nIniciando descarga de {len(df_filtro)} histogramas tipo 2...", flush=True)
        
        # Iteramos sobre las filas filtradas
        for _, fila in df_filtro.iterrows():
            id_original = fila['id']
            
            # Validamos que el ID exista y no esté vacío
            if pd.notna(id_original) and id_original != "":
                
                # Convertimos de forma segura a entero y luego a string para limpiar decimales como '.0'
                try:
                    id_histograma = str(int(float(id_original)))
                except ValueError:
                    id_histograma = str(id_original).strip()
                
                # Definimos el nombre dinámico del archivo CSV para este ID
                nombre_archivo_csv = f"{id_histograma}_histograma.csv"
                
                # Configuramos el payload específico usando la acción 'getloggers' requerida
                payload_histograma = {
                    'action': 'getloggers',
                    'username': os.getenv("API_USER"),
                    'password': os.getenv("API_PASSWORD"),
                    'software': 'APIDocumentation',
                    'id': id_histograma
                }
                
                print(f"\nSolicitando histograma para el ID: {id_histograma}...", flush=True)
                
                response_hist = None
                try:
                    response_hist = requests.get(url, params=payload_histograma, timeout=15)
                    
                    if response_hist.status_code == 200:
                        # Parsear el XML directamente desde la respuesta
                        root_hist = ET.fromstring(response_hist.text)
                        data_rows = []
                        
                        # Buscar cada bloque ArrayOfInt dentro del XML
                        for array in root_hist.findall('.//ArrayOfInt'):
                            valores = [int(nodo.text) for nodo in array.findall('int') if nodo.text is not None]
                            
                            # Asegurar que el bloque contiene la pareja (Índice, Valor)
                            if len(valores) == 2:
                                data_rows.append(valores)
                        
                        # Si encontramos datos válidos, creamos el DataFrame y lo guardamos
                        if data_rows:
                            df_hist = pd.DataFrame(data_rows, columns=['Intervalo', 'Frecuencia'])
                            df_hist.to_csv(nombre_archivo_csv, index=False, sep=';', encoding="utf-8-sig")
                            print(f"-> ¡Éxito! Archivo guardado correctamente como '{nombre_archivo_csv}'.", flush=True)
                        else:
                            print(f"-> No se encontraron estructuras 'ArrayOfInt' válidas para el ID {id_histograma}.", flush=True)
                    else:
                        print(f"-> Error en la petición para ID {id_histograma}. Estado: {response_hist.status_code}", flush=True)
                        
                except Exception as e:
                    print(f"-> Error al procesar el archivo XML del ID {id_histograma}: {e}", flush=True)
                
                finally:
                    # Liberamos explícitamente el socket de conexión de red
                    if response_hist is not None:
                        response_hist.close()
                
                # --- CONTROL DE TRÁFICO (RATE LIMIT) ---
                print("Esperando 1 segundo antes de la siguiente consulta...", flush=True)
                time.sleep(1)
                
        print("\n¡Procesamiento de histogramas finalizado por completo!", flush=True)
    else:
        print("\nNo se encontraron grabaciones con recordingType = 2 en este lote.", flush=True)

## Exportacion de estadísticas de fugas

Con la api dataexportapi.ashx se pueden obtener estadísticas de fugas por sitio (leaksitelist) o general (leakreport). No se devuelven en XML sino en csv directamente.

- El primero devuelve la situación de todos los loggers para el dia de última comunicación.
- El segundo solo devuelve los que se encuentran en fuga, también para el dia de última comunicación.

Ambos incluyen comentarios de los buscafugas y anotaciones.

In [ ]:
## Descarga estructura informe para leaksitelist

# Nueva URL del endpoint
url = "https://api.permanetweb-grupomejoras.com/datagate/api/dataexportapi.ashx"

# Defino el payload con las credenciales y acción
# No incluyo la extraccion explícita de fugas en el payload con 'leakstatus': 'true'
# porque Mejoras ya lo configuró como booleano en el canal 1, con 0 o null para noleak y 1 para leak.
payload = {
    'action': 'getalarm',
    'username': os.getenv("API_USER"),
    'password': os.getenv("API_PASSWORD"),
    'software': 'APIDocumentation',
    'export': 'leaksitelist'
}

# Realizo la petición GET enviando los parámetros
response = requests.get(url, params=payload)

# Controlo la respuesta
if response.status_code == 200:
    try:
        # Definimos la cabecera exactamente con tus campos separados por comas
        # Añadimos \n al final para que los datos de la API empiecen en la siguiente línea
        cabecera = "SiteID,Direccion,Num_SMS,Num_serie,Zona,Latitud_Y,Longitud_X,X,Y,Notas,Fuga,Bateria,Noise,Spread,Threshold,Filter,Low_Cutoff,High_Cutoff,Sensor_type,Leak_Days_duration,Signal,Fecha_ult_com,Internal_status,Fecha_internal_status,Ult_Comentario,Fecha_ult_coment,Comentador,Logger_mode,Fecha_instal,LeakDate_vacio,Fecha_ultima_fuga,EsasyID,Status_Duration,Overdue_Status,LNS_Check,Cuenta_zona,UTC,LoggerType,Last Call-in\n"
        
        # Separamos el contenido de la API en el primer salto de línea (\n)
        # El parámetro 1 indica que solo corte una vez (separando la fila vieja del resto de datos)
        partes = response.content.split(b'\n', 1)
        
        # El resto de datos reales empieza a partir de la posición [1]
        datos_utiles = partes[1] if len(partes) > 1 else b""
        
        # Guardamos el archivo combinando la nueva cabecera y los datos limpios
        with open('reporte_leaksitelist.csv', 'wb') as f:
            f.write(cabecera.encode('utf-8'))
            f.write(datos_utiles)
            
        print("¡Archivo 'reporte_leaksitelist.csv' guardado con cabecera personalizada.")
        
    except Exception as e:
        print(f"Error al procesar y escribir el archivo CSV: {e}")
else:
    print(f"Error en la petición. Código de estado: {response.status_code}")

In [ ]:
## Descarga estructura informe para leakreport

# Nueva URL del endpoint
url = "https://api.permanetweb-grupomejoras.com/datagate/api/dataexportapi.ashx"

# Defino el payload con las credenciales y acción
# No incluyo la extraccion explícita de fugas en el payload con 'leakstatus': 'true'
# porque Mejoras ya lo configuró como booleano en el canal 1, con 0 o null para noleak y 1 para leak.
payload = {
    'action': 'getalarm',
    'username': os.getenv("API_USER"),
    'password': os.getenv("API_PASSWORD"),
    'software': 'APIDocumentation',
    'export': 'leakreport'
}

# Realizo la petición GET enviando los parámetros
response = requests.get(url, params=payload)

# Controlo la respuesta
if response.status_code == 200:
    try:
        # Definimos la cabecera exactamente con tus campos separados por comas
        # Añadimos \n al final para que los datos de la API empiecen en la siguiente línea
        cabecera = "SiteID,Direccion,Num_SMS,Num_serie,Zona,Latitud_Y,Longitud_X,X,Y,Notas,Fuga,Bateria,Noise,Spread,Threshold,Filter,Low_Cutoff,High_Cutoff,Sensor_type,Leak_Days_duration,Signal,Fecha_ult_com,Internal_status,Fecha_internal_status,Ult_Comentario,Fecha_ult_coment,Comentador,Logger_mode,Fecha_instal,LeakDate_vacio,Fecha_ultima_fuga,EsasyID,Status_Duration,Overdue_Status,LNS_Check,Cuenta_zona,UTC,LoggerType,Last Call-in\n"
        
        # Separamos el contenido de la API en el primer salto de línea (\n)
        # El parámetro 1 indica que solo corte una vez (separando la fila vieja del resto de datos)
        partes = response.content.split(b'\n', 1)
        
        # El resto de datos reales empieza a partir de la posición [1]
        datos_utiles = partes[1] if len(partes) > 1 else b""
        
        # Guardamos el archivo combinando la nueva cabecera y los datos limpios
        with open('reporte_leakreport.csv', 'wb') as f:
            f.write(cabecera.encode('utf-8'))
            f.write(datos_utiles)
            
        print("¡Archivo 'reporte_leakreport.csv' guardado con cabecera personalizada.")
        
    except Exception as e:
        print(f"Error al procesar y escribir el archivo CSV: {e}")
else:
    print(f"Error en la petición. Código de estado: {response.status_code}")

# Exportacion de correlaciones

Esta API obtiene y exporta información de autocorrelación dentro de un rango de fechas determinado y una cuenta. También se puede especificar un estado: “Todos”, “Fuga en cualquiera de los lados”, “Fuga en ambos lados”.

La información devuelta coincide con los filtros equivalentes seleccionados en la página “Correlación” de la interfaz de usuario de PermaNET

## Extraigo historico completo de correlaciones hasta la fecha

No se debe automatizar, se usará solo para la carga inicial

In [ ]:
## No incluir, se usa solo para carga inicial

## Exportacion informe de correlacion inicial, con TODOS las correlaciones

# Nueva URL del endpoint
url = "https://api.permanetweb-grupomejoras.com/datagate/api/export/correlations"

# Defino el header con las credenciales y acción
# El resto va en el payload.
headers = {
    'Content-Type': 'application/json'
}

# period = 0 solo saca los valores del dia. period = 4 saca todos, tarda bastante: usar solo para crear fichero inicial, luego incremental.
# status permite definir si se correlacionan todos, los que presentan fuga en al menos un punto, o solo los que presentan fuga entre dos puntos.
payload = {
    'username': os.getenv("API_USER"),
    'password': os.getenv("API_PASSWORD"),
    'software': 'APIDocumentation', 
    'export': 'correlations',
    'period': '4',
    'status': 'all'
}

# Realizo la petición POST enviando los parámetros
response = requests.post(url, headers=headers, json=payload)

# Controlo la respuesta
if response.status_code == 200:
    try:
        # Volcar el contenido crudo directamente a un archivo .csv
        # Usamos 'wb' para escribir en binario y mantener la integridad exacta de los datos
        with open('correlations_historico.csv', 'wb') as f:
            f.write(response.content)
            
        print("¡Archivo completo 'correlations_historico.csv' guardado con éxito")
        
    except Exception as e:
        print(f"Error al escribir el archivo CSV: {e}")
else:
    print(f"Error en la petición. Código de estado: {response.status_code}")

In [ ]:
# Este archivo correlations_historico.csv tiene los registros en orden inverso.
# Hago un script auxiliar para invertirlo.

archivo = 'correlations_historico.csv'

# 1. Leemos todas las líneas del archivo actual
with open(archivo, 'r', encoding='utf-8') as f:
    lineas = f.readlines()

if len(lineas) > 1:
    cabecera = lineas[0]          # Guardamos la primera línea (columnas)
    datos = lineas[1:]            # Separamos las líneas de datos
    datos.reverse()               # Invertimos el orden de los datos en memoria

    # 2. Sobreescribimos el archivo con el nuevo orden
    with open(archivo, 'w', encoding='utf-8', newline='') as f:
        f.write(cabecera)         # Escribimos la cabecera primero
        f.writelines(datos)       # Escribimos los datos ya invertidos
        
    print(f"¡El archivo '{archivo}' ha sido invertido correctamente!")
else:
    print("El archivo está vacío o solo contiene la cabecera.")

## Extraigo diariamente las correlaciones realizadas, y si existen las incorporo

Esta parte si se debe automatizar y ejecutar diariamente a partir de las 7am (comunican de 2 a 5am en horario verano)

In [ ]:
## Exportacion informe de correlacion diario incremental

# Nueva URL del endpoint
url = "https://api.permanetweb-grupomejoras.com/datagate/api/export/correlations"

# Defino el header con las credenciales y acción
# El resto va en el payload.
headers = {
    'Content-Type': 'application/json'
}

# period = 0 solo saca los valores del dia. period = 4 saca todos, tarda bastante: usar solo para crear fichero inicial, luego incremental.
# status permite definir si se correlacionan todos, los que presentan fuga en al menos un punto, o solo los que presentan fuga entre dos puntos.
payload = {
    'username': 'JMCuenca',
    'password': 'M3j0r@sEnergeticas',
    'software': 'APIDocumentation', 
    'export': 'correlations',
    'period': '0',
    'status': 'all'
}

archivo_historico = 'correlations_historico.csv'

# Realizo la petición POST enviando los parámetros
response = requests.post(url, headers=headers, json=payload)

if response.status_code == 200:
    # Obtenemos las líneas del CSV decodificadas como texto
    lineas = response.text.splitlines()
    
    # 1. Control de archivo vacío: Si no hay líneas o solo está la cabecera
    if len(lineas) <= 1:
        print("El último día no registra nuevos datos de correlación. Nada que añadir.")
    
    else:
        # Si el archivo maestro NO existe, lo creamos escribiendo todo (cabecera + datos)
        if not os.path.exists(archivo_historico):
            with open(archivo_historico, 'w', encoding='utf-8', newline='') as f:
                f.write(response.text)
            print(f"¡Archivo histórico creado con éxito con {len(lineas) - 1} registros!")
        
        # Si el archivo maestro SÍ existe, saltamos la cabecera y añadimos solo las filas
        else:
            # Unimos las líneas de datos omitiendo la primera (lineas[0] es la cabecera)
            nuevos_datos = "\n".join(lineas[1:]) + "\n"
            
            # Abrimos en modo 'a' (append) para añadir al final sin borrar lo anterior
            with open(archivo_historico, 'a', encoding='utf-8', newline='') as f:
                f.write(nuevos_datos)
            print(f"¡Carga incremental completada! Se añadieron {len(lineas) - 1} nuevos registros.")

else:
    print(f"Error en la petición. Código de estado: {response.status_code}")